# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rish7186/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: One row represents one housing district in California. Each row contains geographic, demographic, and housing-related information for that district, along with its median house value.

Time window: This dataset does not contain a date or timestamp column. Therefore, there is no explicit time window to define or verify.

In [10]:
import duckdb

# Create DuckDB connection
con = duckdb.connect()

# Register the pandas DataFrame as a SQL table
con.register("housing", df)

# Verify the unit of analysis by checking row count
print("Total Rows:")
print(con.sql("""
SELECT COUNT(*) AS total_rows
FROM housing
""").df())

# Verify available fields
print("\nColumns:")
print(df.columns.tolist())

# Check whether the dataset contains any date/time columns
date_cols = [col for col in df.columns if "date" in col.lower() or "time" in col.lower()]

if date_cols:
    print("\nDate/Time Columns:", date_cols)
else:
    print("\nNo date or timestamp column found.")

Total Rows:
   total_rows
0       17000

Columns:
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value']

No date or timestamp column found.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features:
- longitude
- latitude
- housing_median_age
- total_rooms
- total_bedrooms
- population
- households
- median_income

These variables are available before prediction and are used as input features for the model.

Label:
- median_house_value

This is the target variable that the model predicts.

Context:
- None

The dataset does not contain identifier, timestamp, or grouping columns.

Excluded:
- median_house_value (from the feature set)

Reason: It is the prediction target. Using it as an input feature would cause data leakage and produce unrealistic model performance.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb

# Create DuckDB connection
con = duckdb.connect()

# Register dataframe as a SQL table
con.register("housing", df)

# Display the dataset schema
print(con.sql("DESCRIBE housing").df())

# Classify fields
features = [
    "longitude",
    "latitude",
    "housing_median_age",
    "total_rooms",
    "total_bedrooms",
    "population",
    "households",
    "median_income"
]

label = "median_house_value"

print("\nFeatures:")
for col in features:
    print("-", col)

print("\nLabel:")
print("-", label)

print("\nContext:")
print("None")

print("\nExcluded:")
print("- median_house_value (excluded from features to prevent data leakage)")

          column_name column_type null   key default extra
0           longitude      DOUBLE  YES  None    None  None
1            latitude      DOUBLE  YES  None    None  None
2  housing_median_age      DOUBLE  YES  None    None  None
3         total_rooms      DOUBLE  YES  None    None  None
4      total_bedrooms      DOUBLE  YES  None    None  None
5          population      DOUBLE  YES  None    None  None
6          households      DOUBLE  YES  None    None  None
7       median_income      DOUBLE  YES  None    None  None
8  median_house_value      DOUBLE  YES  None    None  None

Features:
- longitude
- latitude
- housing_median_age
- total_rooms
- total_bedrooms
- population
- households
- median_income

Label:
- median_house_value

Context:
None

Excluded:
- median_house_value (excluded from features to prevent data leakage)


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

I verified the data contract by querying the dataset for row counts, duplicate records (grain), missing values, and available columns. Since the California Housing dataset does not contain a date or timestamp column, there is no time window to verify. These queries confirm that the dataset structure is consistent before feature engineering or model development.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb

# Create DuckDB connection
con = duckdb.connect()

# Register the dataframe as a SQL table
con.register("housing", df)

# -----------------------------
# 1. Row count (Counts)
# -----------------------------
print("Row Count:")
print(con.sql("""
SELECT COUNT(*) AS total_rows
FROM housing
""").df())

# -----------------------------
# 2. Grain check (Duplicate rows)
# -----------------------------
print("\nDuplicate Rows:")
print(con.sql("""
SELECT COUNT(*) AS duplicate_rows
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY longitude,
                            latitude,
                            housing_median_age,
                            total_rooms,
                            total_bedrooms,
                            population,
                            households,
                            median_income,
                            median_house_value
           ) AS rn
    FROM housing
)
WHERE rn > 1
""").df())

# -----------------------------
# 3. Missing values
# -----------------------------
print("\nMissing Values:")
print(df.isnull().sum())

# -----------------------------
# 4. Time window check
# -----------------------------
date_cols = [c for c in df.columns if "date" in c.lower() or "time" in c.lower()]

print("\nTime Window Check:")
if date_cols:
    print("Date/Time columns:", date_cols)
else:
    print("No date or timestamp column found.")

Row Count:
   total_rows
0       17000

Duplicate Rows:
   duplicate_rows
0               0

Missing Values:
longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
median_house_value    0
dtype: int64

Time Window Check:
No date or timestamp column found.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset can be used to predict median house values based on housing and demographic characteristics, but it cannot explain why house prices change or establish cause-and-effect relationships. It does not include factors such as crime rates, school quality, local economic conditions, interest rates, or recent market trends. The dataset also contains missing values in the total_bedrooms column, which may affect the analysis if not handled properly. Since there is no date or timestamp column, the dataset cannot be used to analyze historical trends or changes in house prices over time.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb

# Create DuckDB connection
con = duckdb.connect()

# Register dataframe as a SQL table
con.register("housing", df)

# Dataset size
print("Dataset Size:")
print(con.sql("""
SELECT COUNT(*) AS total_rows
FROM housing
""").df())

# Missing values
print("\nMissing Values:")
print(df.isnull().sum())

# Check for date/time columns
date_cols = [c for c in df.columns if "date" in c.lower() or "time" in c.lower()]

print("\nTime Window Check:")
if date_cols:
    print("Date/Time columns:", date_cols)
else:
    print("No date or timestamp column found.")

Dataset Size:
   total_rows
0       17000

Missing Values:
longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
median_house_value    0
dtype: int64

Time Window Check:
No date or timestamp column found.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.